# Connecticut Overdose Data Exploration

Descriptive exploration and preprocessing of Connecticut accidental drug-related death records.


In [ ]:
# Install the osmnx library


#Import necesassary libariers
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import numpy as np


# 1.2 Loading the dataset

In [ ]:
from pathlib import Path

DATA_PATH = Path("data/accidental_drug_related_deaths.csv")
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download the CSV linked in the README."
    )

death_df = pd.read_csv(DATA_PATH)
df = death_df.copy()
print(f"Loaded {len(death_df):,} records and {len(death_df.columns)} columns.")


# 1.3. Understanding the Dataset

In [ ]:
death_df.head(10)

In [ ]:
death_df.shape

In [ ]:
death_df.info

In [ ]:
death_df.describe

# 1.4 Visualize the Data before preprocessing

In [ ]:
# Visualize missing values using heatmap (Blue shows missing values)
plt.figure(figsize=(10, 6))
sns.heatmap(death_df.isnull(), cbar=False, cmap='RdBu')
plt.title("Missing Values Heatmap - before preprocessing")
plt.show()

# Boxplot for Age
plt.subplot(1, 2, 2)  # 1 row, 2 columns, 2nd subplot
sns.boxplot(y=death_df['Age'], palette=['skyblue', 'lightcoral'])
plt.title('Boxplot of Age')

plt.tight_layout()
plt.show()

# 1.5 More Specific and Map-based Visuals

In [ ]:
import pandas as pd
import plotly.express as px
import re
import numpy as np


def extract_coordinates(location_string):
    """Extracts latitude and longitude from a string using regex."""
    if not isinstance(location_string, str):
        return None, None
    match = re.search(r'\((-?\d+\.?\d*),\s*(-?\d+\.?\d*)\)', location_string)
    if match:
        try:
          latitude = float(match.group(1))
          longitude = float(match.group(2))
          return latitude, longitude
        except ValueError:
          return None, None
    else:
        return None, None


def process_column(df, col, title):
    """Processes a single column by extracting coordinates and plotting a map."""
    cities = df[col].dropna()
    if cities.empty:
      print(f"No non-NaN values found in '{col}' to plot.")
      return

    df_cities = cities.value_counts().reset_index()
    df_cities.columns = ['location', 'count']

    latitudes = []
    longitudes = []
    for location_string in df_cities['location']:
          lat, lon = extract_coordinates(location_string)
          latitudes.append(lat)
          longitudes.append(lon)


    df_cities['lat'] = latitudes
    df_cities['lon'] = longitudes
    df_cities_clean = df_cities.dropna(subset=['lat', 'lon']) #Drop rows if lat or lon are nan
    if not df_cities_clean.empty:
        fig = px.scatter_mapbox(
            df_cities_clean,
            lat='lat',
            lon='lon',
            size='count',
            zoom=7,
            mapbox_style="carto-positron",
            title=title,
        )
        fig.update_layout(
            margin={"r": 0, "t": 40, "l": 0, "b": 0},
            width=1000,
            height=600,
        )
        fig.show()
    else:
       print(f"No valid locations to plot for '{col}'.")


# Main loop for processing each column
for col, title in [('DeathCityGeo', 'Map of Deaths by City'),
                    ('ResidenceCityGeo', 'Map of Residence by City'),
                    ('InjuryCityGeo', 'Map of Injuries by City')]:
    process_column(death_df, col, title)

# 2. Pie chart for Sex
sex_counts = death_df['Sex'].dropna().value_counts()
if not sex_counts.empty:
    plt.figure(figsize=(8, 6))
    plt.pie(sex_counts, labels=sex_counts.index, autopct='%1.1f%%', startangle=140, colors=['pink', 'skyblue'])
    plt.title('Distribution of Sex')
    plt.show()
else:
    print("No non-NaN values found in 'Sex' to plot")

# 3. Stacked bar chart for Race
race_counts = death_df['Race'].dropna().value_counts()
if not race_counts.empty:
    plt.figure(figsize=(10, 6))
    race_counts.plot(kind='bar', stacked=True, color=sns.color_palette('Set2', len(race_counts)))
    plt.title('Distribution of Race')
    plt.xticks(rotation=45, ha="right")
    plt.show()
else:
  print("No non-NaN values found in 'Race' to plot")

# 4. Bar chart for Residence City
residence_city_counts = death_df['Residence City'].dropna().value_counts().head(10) # show only top 10
if not residence_city_counts.empty:
    plt.figure(figsize=(12, 6))
    residence_city_counts.plot(kind='bar', color='teal')
    plt.title('Top 10 Residence Cities')
    plt.xticks(rotation=45, ha="right")
    plt.show()
else:
    print("No non-NaN values found in 'Residence City' to plot")

# 6. Bar Chart of Fentanyl use per Age
if 'Fentanyl' in death_df.columns and not death_df['Fentanyl'].dropna().empty and not death_df['Age'].dropna().empty:
    age_fentanyl = death_df.groupby('Age')['Fentanyl'].count().dropna() # Count non-null values
    if not age_fentanyl.empty:
        plt.figure(figsize=(12,6))
        plt.bar(age_fentanyl.index, age_fentanyl.values, color='purple')
        plt.title('Fentanyl Use Count by Age')
        plt.xlabel('Age')
        plt.ylabel('Fentanyl Use (Count)')
        plt.show()
    else:
       print("No non-NaN values for both 'Age' and 'Fentanyl' columns to plot.")
else:
   print("Either 'Fentanyl' or 'Age' column is missing or empty, cannot plot bar chart.")


# --- END OF VISUALIZATIONS ---

1.6 Checking which values have similar, insignificant or poorly differentiated categories

In [ ]:
#Gettting an idea for the unique values in each feature
for column in death_df.columns:
    print(f"**Column:** `{column}`")
    value_counts = death_df[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))

    print("-" * 40)

# 2. Preprocessing

2.1 Encoding, Discretization and Merging Categories.

In [ ]:
def replace_drug_values(df, drug_cols):
    """Replaces 'Y' with 1 and NaN with 0 in specified drug-related columns.

    Args:
        df (pd.DataFrame): The input DataFrame.
        drug_cols (list): List of column names to process.

    Returns:
        pd.DataFrame: The DataFrame with replaced values.
    """
    df_copy = df.copy()
    for col in drug_cols:
        df_copy[col] = df_copy[col].replace('Y', 1).fillna(0)
    return df_copy

def encode_sex_and_discretize_age(df):
    """
    Encodes 'Sex' (Male/Female to 1/0) and discretizes 'Age' into 10-year bins.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The DataFrame with encoded 'Sex' and binned 'Age'.
    """
    df_copy = df.copy()
    print(f"  Shape before Sex cleaning: {df_copy.shape}")

    # Clean 'Sex' column
    df_copy['Sex'] = df_copy['Sex'].str.strip().str.lower()

    # Drop rows with NaN, 'Unknown', or 'X' in 'Sex'
    df_copy = df_copy[~df_copy['Sex'].isin([np.nan, 'unknown', 'x'])]
    print(f"  Shape after dropping invalid Sex values: {df_copy.shape}")


    # Encode 'Sex'
    df_copy['Sex'] = df_copy['Sex'].replace({'male': 1, 'female': 0}, inplace = False)


    # Ensure 'Age' column is numeric and coerce errors
    df_copy['Age'] = pd.to_numeric(df_copy['Age'], errors='coerce')

    # Drop rows where age is NaN after coercion
    print(f"  Shape before dropping NaN Age: {df_copy.shape}")
    df_copy = df_copy.dropna(subset=['Age'])
    print(f"  Shape after dropping NaN Age: {df_copy.shape}")

    # Discretize Age into 10 year bins
    min_age = df_copy['Age'].min()
    max_age = df_copy['Age'].max()
    bins = np.arange(min_age, max_age + 10, 10) # Ensure the last bin includes max age
    df_copy['Age_Bin'] = pd.cut(df_copy['Age'], bins=bins, labels=False, right = False) # right = False for the left inclusive bins
    print(f"  Unique values in Age_Bin after binning: {df_copy['Age_Bin'].unique()}")
    return df_copy

def merge_and_encode_race_ethnicity(df, race_col='Race', ethnicity_col='Ethnicity'):
    """
    Merges race and ethnicity columns into a simplified 'Race_Ethnicity' column,
    categorizing into 'White', 'Black', 'Asian', 'Other' for race and
    'Hispanic/Latino', 'Not Hispanic/Latino', and 'Unknown' for ethnicity,
    and then encodes the resulting categories numerically.

    Args:
        df (pd.DataFrame): Input DataFrame.
        race_col (str): Name of the race column.
        ethnicity_col (str): Name of the ethnicity column.

    Returns:
        pd.DataFrame: DataFrame with merged and encoded 'Race_Ethnicity' column.
    """

    df_copy = df.copy()

    # Handle missing columns by creating them if not there
    if race_col not in df_copy.columns:
      df_copy[race_col] = None
    if ethnicity_col not in df_copy.columns:
      df_copy[ethnicity_col] = None

    #Ensure columns are strings
    df_copy[race_col] = df_copy[race_col].astype(str)
    df_copy[ethnicity_col] = df_copy[ethnicity_col].astype(str)

    # Standardize values by making lowercase and stripping whitespace
    df_copy[race_col] = df_copy[race_col].str.strip().str.lower()
    df_copy[ethnicity_col] = df_copy[ethnicity_col].str.strip().str.lower()


    def combine_race_ethnicity(row):
        race = row[race_col]
        ethnicity = row[ethnicity_col]

        # Simplify ethnicity first
        if 'hispanic' in ethnicity or 'latino' in ethnicity or 'spanish' in ethnicity or 'puerto rican' in ethnicity or 'mexican' in ethnicity or 'cuban' in ethnicity:
              ethnicity_category = 'hispanic/latino'
        elif 'no' in ethnicity or 'not spanish' in ethnicity:
               ethnicity_category = 'not hispanic/latino'
        else:
              ethnicity_category = 'unknown'

        # Simplify race
        if 'white' in race or race == 'white':
            race_category = 'white'
        elif 'black' in race or 'african american' in race or 'haitian' in race:
            race_category = 'black'
        elif 'asian indian' in race or 'indian' in race or 'chinese' in race or 'korean' in race or 'other asian' in race or 'asian' in race:
            race_category = 'asian'
        else:
            race_category = 'other'

        return f"{race_category}_{ethnicity_category}"

    df_copy['Race_Ethnicity'] = df_copy.apply(combine_race_ethnicity, axis=1)

    # Create the encoding
    df_copy['Race_Ethnicity_Encoded'] = df_copy['Race_Ethnicity'].astype('category').cat.codes

    # Drop original columns
    df_copy = df_copy.drop(columns=[race_col, ethnicity_col, 'Race_Ethnicity'], errors = 'ignore')

    return df_copy

def encode_opioid(df):
  """
    Encodes opioid-related columns into a single 'Opioid_Death' column.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: DataFrame with the 'Opioid_Death' column.
    """
  df_copy = df.copy()
  opioid_cols = [col for col in df_copy.columns if 'Opiate' in col or 'Heroin' in col or 'Fentanyl' in col or 'Methadone' in col]

  # Convert opioid columns to numeric, coercing errors to NaN and then fill with 0
  for col in opioid_cols:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce').fillna(0)

  df_copy['Opioid_Death'] = df_copy[opioid_cols].max(axis=1)
  return df_copy

def preprocess_death_data(df):
  """
    Standardizes the yes/no columns to 1s and 0s.
    Drops the column if there is a high degree of missingness.
    Encodes 'Date Type', standardizes and encodes 'Description of Injury',
    and standardizes and encodes 'Cause of Death'.
    Fills all missing values.
    Args:
      df (pd.DataFrame): The input dataframe

    Returns:
      pd.DataFrame: The processed dataframe.
    """
  df_copy = df.copy()
  yes_no_columns = ['Any Opioid', 'Heroin','Cocaine','Fentanyl','Opiate','Tramadol','Benzodiazepine','Methadone','Amphetamine']

  for col in yes_no_columns:
        if col in df_copy.columns:
            df_copy[col] = df_copy[col].replace('Y',1).fillna(0)
  # --- Date Type Processing ---
  if 'Date Type' in df_copy.columns:
    print(f"Unique values in 'Date Type' before encoding: {df_copy['Date Type'].unique()}")
    df_copy['Date Type'] = df_copy['Date Type'].str.lower().fillna('unknown')
    df_copy['Date_Type_Encoded'] = df_copy['Date Type'].astype('category').cat.codes

  # --- Description of Injury Processing ---
  if 'Description of Injury' in df_copy.columns:
      df_copy['Description of Injury'] = df_copy['Description of Injury'].str.lower().fillna('other')
      df_copy['Description of Injury'] = df_copy['Description of Injury'].apply(lambda x: 'substance abuse' if 'use' in x else 'other')
      df_copy['Description_of_Injury_Encoded'] = df_copy['Description of Injury'].astype('category').cat.codes


  # --- Cause of Death Processing ---
  if 'Cause of Death' in df_copy.columns:
      print(f"Unique values in 'Cause of Death' before encoding: {df_copy['Cause of Death'].unique()}")
      df_copy['Cause of Death'] = df_copy['Cause of Death'].str.lower().fillna('other') # fill with 'other' for consistency
      df_copy['Cause of Death'] = df_copy['Cause of Death'].apply(lambda x: 'acute intoxication' if 'tox' in x else 'other')
      df_copy['Cause_of_Death_Encoded'] = df_copy['Cause of Death'].astype('category').cat.codes

  if 'Manner of Death' in df_copy.columns:
          print(f"Unique values in 'Manner of Death' before encoding: {df_copy['Manner of Death'].unique()}")
          df_copy['Manner of Death'] = df_copy['Manner of Death'].str.lower().fillna('unknown')
          df_copy['Manner of Death'] = df_copy['Manner of Death'].replace({
                'accident': 'accident',
                'natural': 'natural',
                'homicide' : 'homicide',
                'suicide' : 'suicide',
                 'pending' : 'pending',
                 'undetermined' : 'undetermined',
                 'acciddent': 'accident'
            })
          df_copy['Manner_of_Death_Encoded'] = df_copy['Manner of Death'].astype('category').cat.codes

  missing_percentage = df_copy.isnull().sum() / len(df_copy)
  columns_to_drop_missingness = missing_percentage[missing_percentage > 0.8].index.tolist()
  df_copy = df_copy.drop(columns=columns_to_drop_missingness, errors='ignore')


  return df_copy

2.2 Imputing missing values, droping records with many missing features and extracting year from date

In [ ]:
def impute_geographic_data_majority_rule(df, city_col='Residence City', state_col='Residence State', county_col='Residence County'):
    """
    Imputes missing city or state using majority rule, imputes county based on city,
    drops rows where both city and state are missing,
    and prints unique counties and corresponding cities.

    Args:
        df (pd.DataFrame): The input DataFrame.
        city_col (str): The name of the city column.
        state_col (str): The name of the state column.
        county_col (str): The name of the county column.

    Returns:
        pd.DataFrame: The DataFrame with imputed values.
    """

    df_copy = df.copy() # Make a copy of the dataframe

    # Create city-to-state and state-to-city mappings
    city_to_state = df_copy.groupby(city_col)[state_col].agg(lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    state_to_city = df_copy.groupby(state_col)[city_col].agg(lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()
    city_to_county = df_copy.groupby(city_col)[county_col].agg(lambda x: x.mode()[0] if not x.mode().empty else None).to_dict()


    rows_to_drop = []
    for index, row in df_copy.iterrows():
        city = row[city_col]
        state = row[state_col]
        county = row[county_col]


        # Drop rows where both city and state are missing
        if pd.isna(city) and pd.isna(state):
            rows_to_drop.append(index)
            continue

        # Majority-rule imputation for state
        if pd.isna(state) and not pd.isna(city) and city in city_to_state:
            df_copy.at[index, state_col] = city_to_state[city]
        elif pd.isna(city) and not pd.isna(state) and state in state_to_city:
            df_copy.at[index, city_col] = state_to_city[state]

        # Majority-rule imputation for county
        if pd.isna(county) and not pd.isna(city) and city in city_to_county:
              df_copy.at[index, county_col] = city_to_county[city]

    # Drop rows where both city and state were originally missing
    df_copy = df_copy.drop(rows_to_drop)

    # Print unique counties and cities
    print("\nUnique Counties and Corresponding Cities:")
    county_city_pairs = df_copy.groupby('Residence County')['Residence City'].unique().to_dict()
    for county, cities in county_city_pairs.items():
        print(f"  County: {county}")
        for city in cities:
            print(f"    - {city}")

    return df_copy

city_to_state_dict = {
    'ALFRED STATION': 'NY',
    'ALLSTON': 'MA',
    'ANDOVER': 'MA',
    'ANSONIA': 'CT',
    'ASTORIA': 'NY',
    'AVENTURA': 'FL',
    'AVON': 'CT',
    'BALTIC': 'CT',
    'BASS HARBOR': 'ME',
    'BEACON FALLS': 'CT',
    'BERLIN': 'CT',
    'BETHEL': 'CT',
    'BOLTON': 'CT',
    'BOZRAH': 'CT',
    'BRANFORD': 'CT',
    'BRIDGEPORT': 'CT',
    'BRISTOL': 'CT',
    'BROAD BROOK': 'CT',
    'BROOKFIELD': 'CT',
    'BROOKLYN': 'NY',
    'BURLINGTON': 'VT',
    'CANAAN': 'CT',
    'CANTERBURY': 'CT',
    'CANTON': 'CT',
    'CHARLESTOWN': 'MA',
    'CHESAPEAKE': 'VA',
    'CHESHIRE': 'CT',
    'CHESTER': 'CT',
    'CLINTON': 'CT',
    'COLCHESTER': 'CT',
    'COLLINSVILLE': 'CT',
    'COVENTRY': 'CT',
    'CROMWELL': 'CT',
    'DANBURY': 'CT',
    'DANIELSON': 'CT',
    'DARIEN': 'CT',
    'DAYVILLE': 'CT',
    'DEEP RIVER': 'CT',
    'DEER LODGE': 'MT',
    'DERBY': 'CT',
    'DOVER PLAINS': 'NY',
    'DURHAM': 'CT',
    'EAST BERLIN': 'CT',
    'EAST CANAAN': 'CT',
    'EAST GRANBY': 'CT',
    'EAST HADDAM': 'CT',
    'EAST HAMPTON': 'CT',
    'EAST HARTFORD': 'CT',
    'EAST HARTLAND': 'CT',
    'EAST HAVEN': 'CT',
    'EAST LYME': 'CT',
    'EAST MEADOW': 'NY',
    'EAST WINDSOR': 'CT',
    'EAST WOODSTOCK': 'CT',
    'EASTFORD': 'CT',
    'ELLINGTON': 'CT',
    'ENFIELD': 'CT',
    'ESSEX': 'CT',
    'FAIRFIELD': 'CT',
    'FALLS VILLAGE': 'CT',
    'FARMINGTON': 'CT',
    'FLUSHING': 'NY',
    'FRANKLIN': 'CT',
    'GALES FERRY': 'CT',
    'GLASTONBURY': 'CT',
    'GOSHEN': 'CT',
    'GRANBY': 'CT',
    'GRANVILLE': 'MA',
    'GREENWICH': 'CT',
    'GROTON': 'CT',
    'GUILFORD': 'CT',
    'HADDAM': 'CT',
    'HAMDEN': 'CT',
    'HAMILTON': 'NY',
    'HAMPTON BAYS': 'NY',
    'HANCOCK': 'CT',
    'HARTFORD': 'CT',
    'HARWINTON': 'CT',
    'HEBRON': 'CT',
    'HOMELESS': 'CT',  # Or leave as None if it is not a real city
    'HURLEY': 'NY',
    'IVORYTON': 'CT',
    'JASPER': 'CT',
    'JEWETT CITY': 'CT',
    'KENSINGTON': 'CT',
    'KILLINGLY': 'CT',
    'KILLINGWORTH': 'CT',
    'KINGSTON': 'NY',
    'LEBANON': 'CT',
    'LEDYARD': 'CT',
    'LISBON': 'CT',
    'LITCHFIELD': 'CT',
    'MADISON': 'CT',
    'MAHOPAC': 'NY',
    'MANCHESTER': 'CT',
    'MANSFIELD': 'CT',
    'MARLBOROUGH': 'CT',
    'MERIDEN': 'CT',
    'MIDDLEBURY': 'CT',
    'MIDDLETOWN': 'CT',
    'MILFORD': 'CT',
    'MILLERTON': 'NY',
    'MONROE': 'CT',
    'MONTVILLE': 'CT',
    'MOODUS': 'CT',
    'MOOSUP': 'CT',
    'MORRIS': 'CT',
    'MYSTIC': 'CT',
    'N HAVEN': 'CT',
    'NAUGATUCK': 'CT',
    'NEW BEDFORD': 'MA',
    'NEW BRIT': 'CT',
    'NEW BRITAIN': 'CT',
    'NEW CASTLE': 'DE',
    'NEW FAIRFIELD': 'CT',
    'NEW HAVEN': 'CT',
    'NEW LONDON': 'CT',
    'NEW MILFORD': 'CT',
    'NEW YORK': 'NY',
    'NEWINGTON': 'CT',
    'NEWTOWN': 'CT',
    'NIANTIC': 'CT',
    'NO HAVEN': 'CT',
    'NORTH BRANFORD': 'CT',
    'NORTH CANAAN': 'CT',
    'NORTH HAVEN': 'CT',
    'NORTH SALEM': 'NY',
    'NORTH STONINGTON': 'CT',
    'NORTHFORD': 'CT',
    'NORWALK': 'CT',
    'NORWICH': 'CT',
    'OAKDALE': 'CT',
    'OAKVILLE': 'CT',
    'OLD GREENWICH': 'CT',
    'OLD LYME': 'CT',
    'OLD SAYBROOK': 'CT',
    'ONECO': 'CT',
    'ORANGE': 'CT',
    'OSHKOSH': 'WI',
    'OXFORD': 'CT',
    'PALMER': 'MA',
    'PAWCATUCK': 'CT',
    'PAWTUCKET': 'RI',
    'PITTSBURGH': 'PA',
    'PLAINFIELD': 'CT',
    'PLAINVILLE': 'CT',
    'PLANTSVILLE': 'CT',
    'PLYMOUTH': 'CT',
    'POMFRET': 'CT',
    'PORT CHESTER': 'NY',
    'PORTCHESTER': 'NY',
    'PORTLAND': 'CT',
    'PRESTON': 'CT',
    'PROSPECT': 'CT',
    'PUTNAM': 'CT',
    'QUAKER HILL': 'CT',
    'QUINCY': 'MA',
    'REDDING': 'CT',
    'RIDGEFIELD': 'CT',
    'ROCKVILLE': 'CT',
    'ROCKY HILL': 'CT',
    'ROGERS': 'CT',
    'RYE': 'NY',
    'S GLASTONBURY': 'CT',
    'SAKEM': 'CT',
    'SALEM': 'CT',
    'SALISBURY': 'CT',
    'SANDY HOOK': 'CT',
    'SANTA CRUZ': 'CA',
    'SARATOGA SPGS': 'NY',
    'SEYMOUR': 'CT',
    'SHARON': 'CT',
    'SHELTON': 'CT',
    'SHERMAN': 'CT',
    'SIMSBURY': 'CT',
    'SOMERS': 'CT',
    'SOUTH WINDSOR': 'CT',
    'SOUTHBURY': 'CT',
     'SOUTHINGTON': 'CT',
    'STAFFORD': 'CT',
    'STAFFORD SPGS': 'CT',
    'STAFFORD SPRINGS': 'CT',
    'STAMFORD': 'CT',
    'STANFORDVILLE': 'NY',
    'STERLING': 'CT',
    'STRATFORD': 'CT',
    'SUDBURY': 'MA',
    'SUFFIELD': 'CT',
    'TACONIC': 'CT',
    'TARIFFVILLE': 'CT',
    'TERRYVILLE': 'CT',
    'THOMASTON': 'CT',
    'TOLLAND': 'CT',
    'TORRINGTON': 'CT',
    'TRUMBULL': 'CT',
    'UNCASVILLE': 'CT',
    'UNIONVILLE': 'CT',
    'UNKNOWN': None,  # Or leave as None if it is not a real city
    'VERNON': 'CT',
    'VINE GROVE': 'KY',
    'VOLUNTOWN': 'CT',
    'W HAVEN': 'CT',
    'WALLINGFORD': 'CT',
    'WASHINGTON': 'DC',
     'WATERBRUY': 'CT',
    'WATERBURY': 'CT',
    'WATERFORD': 'CT',
    'WATERTOWN': 'CT',
    'WEATHERSFIELD': 'CT',
    'WEATOGUE': 'CT',
    'WELLESLEY': 'MA',
    'WEST HARTFORD': 'CT',
    'WEST HAVEN': 'CT',
    'WESTBROOK': 'CT',
    'WESTPORT': 'CT',
    'WETHERSFIELD': 'CT',
    'WILLACOOCHEE': 'GA',
    'WILLIMANTIC': 'CT',
    'WILLINGTON': 'CT',
    'WILTON': 'CT',
    'WINCHESTER': 'CT',
    'WINDHAM': 'CT',
    'WINDSOR': 'CT',
    'WINDSOR LOCKS': 'CT',
     'WINGDALE': 'NY',
    'WINSTED': 'CT',
    'WOLCOTT': 'CT',
    'WOODBRIDGE': 'CT',
    'WOODBURY': 'CT',
    'ZIONSVILLE': 'IN'
}

def impute_geographic_data(df, city_col='Residence City', state_col='Residence State'):
    """
    Imputes missing state based on city using a dictionary lookup, and drops rows where both are missing.

    Args:
        df (pd.DataFrame): The input DataFrame.
        city_col (str): The name of the city column.
        state_col (str): The name of the state column.

    Returns:
        pd.DataFrame: The DataFrame with imputed values.
    """
    df_copy = df.copy()
    rows_to_drop = []
    for index, row in df_copy.iterrows():
        city = row[city_col]
        state = row[state_col]

        # Drop rows where both city and state are missing
        if pd.isna(city) and pd.isna(state):
          rows_to_drop.append(index)
          continue

        # Impute state based on city
        if pd.isna(state) and not pd.isna(city) and city in city_to_state_dict:
          df_copy.at[index, state_col] = city_to_state_dict[city]

    df_copy = df_copy.drop(rows_to_drop)
    return df_copy

def clean_and_impute(df):
    """
    Removes rows with missing Age, Sex, or Manner of Death;
    Drops the 'Ethnicity' column due to majority missing values;
    Imputes NaN values for Description of Injury with 'Unknown';
    Imputes NaN values for Race with 'Other'.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The processed DataFrame.
    """

    df_copy = df.copy() # Make a copy of the dataframe

    # 1. Remove rows with missing 'Age', 'Sex', or 'Manner of Death'
    df_copy = df_copy.dropna(subset=['Age', 'Sex', 'Manner of Death', 'Residence State','Residence City'])

    # 2. Drop 'Ethnicity' column due to majority missing values.
    # It will not benefit any model or analysis and cannot be assumed or judged ethically
    df_copy = df_copy.drop(columns=['Ethnicity'], errors='ignore')

    # 3. Impute NaN values for 'Description of Injury' with 'Unknown'
    df_copy['Description of Injury'] = df_copy['Description of Injury'].fillna('Unknown')

    # 4. Impute NaN values for 'Race' with 'Other'
    df_copy['Race'] = df_copy['Race'].fillna('Other')


    return df_copy
def process_dates(df):
    """
    Processes date columns to extract just the year and renames 'Date' to 'Year'.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: The DataFrame with processed date columns.
    """
    df_copy = df.copy()
    date_cols = ['Date', 'Date of Death', 'Date Reported', 'Injury Date']
    for col in date_cols:
        if col in df_copy.columns:
            # Convert to datetime, coerce errors to NaT
            df_copy[col] = pd.to_datetime(df_copy[col], errors='coerce')
            # Extract year
            df_copy[col] = df_copy[col].dt.year
    # Rename 'Date' to 'Year'
    if 'Date' in df_copy.columns:
      df_copy = df_copy.rename(columns={'Date': 'Year'})
    return df_copy

2.3 Dealing with outliers

In [ ]:
def remove_outliers_iqr(df, column):
    """
    Removes outliers from a specified column using the IQR method.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column to check for outliers.

    Returns:
        pd.DataFrame: The DataFrame with outliers removed.
    """
    # Check if the column is numeric
    if df[column].dtype not in ['int64', 'float64']:
        return df # If it's not numeric skip

def remove_outliers_iqr(df, column):
    """
    Removes outliers from a specified column using the IQR method.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column to check for outliers.

    Returns:
        pd.DataFrame: The DataFrame with outliers removed.
    """
    # Check if the column is numeric
    if df[column].dtype not in ['int64', 'float64']:
        return df # If it's not numeric skip

    # Check if the column is binary (has only 2 unique values)
    if df[column].nunique() <= 2:
        return df # If it's binary skip

    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outlier_indices = df[(df[column] < lower_bound) | (df[column] > upper_bound)].index
    df_cleaned = df.drop(outlier_indices)
    if df_cleaned.empty:
        print(f"Warning: DataFrame is empty after attempting to remove outliers in column '{column}'. Skipping outlier removal for this column.")
        return df # Return original df if empty
    return df_cleaned


2.4 Droping useless columns that contribute poor or over-specialized data

In [ ]:
# %%
def process_drug_deaths_data(df_replaced):
    """
    Processes a drug deaths dataset by dropping unnecessary columns
    and keeping only relevant geographic information.

    Args:
        df (pd.DataFrame): The input DataFrame containing drug deaths data.

    Returns:
        pd.DataFrame: The processed DataFrame.
    """

    # 1. Drop Location-Related Columns (Except Residence)
    # Justification: We are primarily interested in the residence location of the deceased.
    # Injury and death locations are not essential for our analysis, and keeping them
    # would add complexity without significant benefit.
    columns_to_drop_location = [
        'Residence County','Injury City', 'Injury County', 'Injury State', 'Injury Place',
        'Death City', 'Death County', 'Death State', 'Location',
        'Location if Other', 'ResidenceCityGeo', 'InjuryCityGeo', 'DeathCityGeo'
    ]

    # Store the columns before the drop
    columns_before_drop_location = df_replaced.columns.tolist()
    df_replaced = df_replaced.drop(columns=columns_to_drop_location, errors='ignore')
    # Find the columns that were dropped
    columns_dropped_location = set(columns_before_drop_location) - set(df_replaced.columns.tolist())
    print('Columns dropped (location): ', list(columns_dropped_location), flush=True) # ADDED flush=True



    # 2. Drop Columns with Low Variance or Redundancy
    # Justification: Some columns might have very little variation (e.g., almost all values are the same)
    # or might be redundant (e.g., 'Heroin' and 'Heroin death certificate (DC)').
    # These columns don't provide much information for analysis and can be dropped.
    columns_to_drop_low_variance = [
        'Heroin death certificate (DC)', # Redundant with 'Heroin'
        'Opiate NOS', # Likely low variance or vague
        'Heroin/Morph/Codeine' # Specific combination, might be low occurrence
    ]
    # Store columns before dropping
    columns_before_drop_low_variance = df_replaced.columns.tolist()
    df_replaced = df_replaced.drop(columns=columns_to_drop_low_variance, errors='ignore')
    # Find dropped columns
    columns_dropped_low_variance = set(columns_before_drop_low_variance) - set(df_replaced.columns.tolist())
    print('Columns dropped (low variance): ', list(columns_dropped_low_variance), flush=True) # ADDED flush=True


    # 3. Drop Columns with High Missingness (if needed)
    # Justification: If a column has a very high percentage of missing values, it might not be useful.
    # You would need to analyze the missingness in your specific dataset and set a threshold.
    # Example (replace '0.8' with your threshold):
    missing_percentage = df_replaced.isnull().sum() / len(df_replaced)
    columns_to_drop_missingness = missing_percentage[missing_percentage > 0.8].index.tolist()
     # Store columns before dropping
    columns_before_drop_missingness = df_replaced.columns.tolist()
    df_replaced = df_replaced.drop(columns=columns_to_drop_missingness, errors='ignore')
    # Find dropped columns
    columns_dropped_missingness = set(columns_before_drop_missingness) - set(df_replaced.columns.tolist())
    print('Columns dropped (high missingness): ', list(columns_dropped_missingness), flush=True) # ADDED flush=True


    # 4. Drop 'Other' Column (if not useful)
    # Justification: The 'Other' column might contain miscellaneous or poorly defined information.
    # If it doesn't provide valuable insights, it's better to drop it.
    print(f"Columns in df_replaced before 'Other' check: {df_replaced.columns.tolist()}", flush=True) # ADDED flush=True

    # Store columns before dropping
    columns_before_drop_other = df_replaced.columns.tolist()
    df_replaced = df_replaced.drop(columns=['Other'], errors='ignore')
    #Find dropped columns
    columns_dropped_other = set(columns_before_drop_other) - set(df_replaced.columns.tolist())
    print('Columns dropped (other): ', list(columns_dropped_other), flush=True) # ADDED flush=True

    return df_replaced


2.5 Ensuring smooth control flow and keep track of function calls and dataframe manipulation.

In [ ]:
def process_data(df):
    """Processes the dataframe through all the defined steps and returns a final dataframe or None if an intermediate dataframe is empty"""
    drug_cols = [col for col in df.columns if "Any" in col or "Heroin" in col or "Cocaine" in col or "Fentanyl" in col or "Opiate" in col or "Tramadol" in col or "Benzodiazepine" in col or "Methadone" in col or 'Amphetamine' in col or 'Ethanol' in col]

    print(f"Starting data processing with shape: {df.shape}")

    df_processed_drugs = replace_drug_values(df.copy(), drug_cols)
    print(f"df_processed_drugs shape after replace_drug_values: {df_processed_drugs.shape}")
    if df_processed_drugs.empty:
        print("Error: DataFrame is empty after replace_drug_values.")
        return None

    df_processed_location = process_drug_deaths_data(df_processed_drugs)
    print(f"df_processed_location shape after process_drug_deaths_data: {df_processed_location.shape}")
    if df_processed_location.empty:
        print("Error: DataFrame is empty after process_drug_deaths_data.")
        return None

    df_processed_dates = preprocess_death_data(df_processed_location)
    print(f"df_processed_dates shape after preprocess_death_data: {df_processed_dates.shape}")
    if df_processed_dates.empty:
            print("Error: DataFrame is empty after preprocess_death_data.")
            return None

    df_imputed_geo = impute_geographic_data(df_processed_dates)
    print(f"df_imputed_geo shape after impute_geographic_data: {df_imputed_geo.shape}")
    if df_imputed_geo.empty:
        print("Error: DataFrame is empty after impute_geographic_data.")
        return None

    df_cleaned_imputed = clean_and_impute(df_imputed_geo)
    print(f"df_cleaned_imputed shape after clean_and_impute: {df_cleaned_imputed.shape}")
    if df_cleaned_imputed.empty:
        print("Error: DataFrame is empty after clean_and_impute.")
        return None

    df_encoded_race_eth = merge_and_encode_race_ethnicity(df_cleaned_imputed.copy())
    print(f"df_encoded_race_eth shape after merge_and_encode_race_ethnicity: {df_encoded_race_eth.shape}")
    if df_encoded_race_eth.empty:
        print("Error: DataFrame is empty after merge_and_encode_race_ethnicity.")
        return None

    df_encoded_sex_age = encode_sex_and_discretize_age(df_encoded_race_eth.copy())
    print(f"df_encoded_sex_age shape after encode_sex_and_discretize_age: {df_encoded_sex_age.shape}")
    if df_encoded_sex_age.empty:
        print("Error: DataFrame is empty after encode_sex_and_discretize_age.")
        return None

    df_encoded_opioid = encode_opioid(df_encoded_sex_age.copy())
    print(f"df_encoded_opioid shape after encode_opioid: {df_encoded_opioid.shape}")
    if df_encoded_opioid.empty:
        print("Error: DataFrame is empty after encode_opioid.")
        return None

    df_processed_dates_2 = process_dates(df_encoded_opioid)
    print(f"df_processed_dates_2 shape after process_dates: {df_processed_dates_2.shape}")
    if df_processed_dates_2.empty:
        print("Error: DataFrame is empty after process_dates.")
        return None

    numeric_columns = df_processed_dates_2.select_dtypes(include=np.number).columns
    df_cleaned_outliers = df_processed_dates_2.copy()

    for column in numeric_columns:
        df_cleaned_outliers = remove_outliers_iqr(df_cleaned_outliers, column)
        if df_cleaned_outliers.empty:
              print(f"Warning: DataFrame is empty after remove_outliers_iqr in column '{column}'.")
              break

    df_final = df_cleaned_outliers  # Final data set
    print(f"Finished processing with final shape: {df_final.shape}")
    return df_final

# Apply the function
df_final = process_data(death_df.copy())

if df_final is not None:
      # Output the final DataFrame
      print("Final DataFrame (df_final) after all transformations:")
      print(df_final.head())
      print(df_final.shape)
      print(df_final.info())


2.6 Download the cleaned dataset

In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)
processed_path = output_dir / "exploration_processed_records.csv"
if "df_final" not in locals() or not isinstance(df_final, pd.DataFrame):
    raise RuntimeError("Run the preprocessing cells before saving the processed data.")
df_final.to_csv(processed_path, index=False)
print(f"Saved processed records to {processed_path}")


2.6 Visualizing changes in the dataset (missing values, unique values and outliers)

In [ ]:
# 1. Visualize Missing Values
plt.figure(figsize=(10, 6))
sns.heatmap(df_final.isnull(), cbar=False, cmap='RdBu')
plt.title("Missing Values Heatmap (After Preprocessing)")
plt.show()

print("\nMissing values per column:")
print(df_final.isnull().sum())
print(f"\nTotal missing values: {df_final.isnull().sum().sum()}")

# 2. Visualize Outliers (Boxplots) for Year and Age
plt.figure(figsize=(10, 5))

# Boxplot for Year
plt.subplot(1, 2, 1)  # 1 row, 2 columns, 1st subplot
sns.boxplot(y=df_final['Year'], palette=['Olive'])
plt.title('Boxplot of Year')

# Boxplot for Age
plt.subplot(1, 2, 2)  # 1 row, 2 columns, 2nd subplot
sns.boxplot(y=df_final['Age'], palette=['Teal'])
plt.title('Boxplot of Age')

plt.tight_layout()
plt.show()

# Print unique values and their counts for each column
for column in df_final.columns:
    print(f"**Column:** `{column}`")
    value_counts = df_final[column].value_counts()
    formatted_values = [f"{value} ({count})" for value, count in value_counts.items()]
    print("  Unique Values:", ", ".join(formatted_values))
    print("-" * 40)

# 3. Visualzation - After Preprocessing

3.1 General Visuals - Understanding the opioid crisis

In [ ]:
# --- Visualization 1: Opioid Deaths by Year ---
def visualize_opioid_deaths_by_year(df):
    """
    Visualizes the number of opioid deaths per year with an average line.

    Args:
        df (pd.DataFrame): Input DataFrame containing opioid death data.
    """
    # Group data by 'Year' and sum 'Opioid_Death' to count deaths per year
    opioid_deaths_by_year = df.groupby('Year')['Opioid_Death'].sum().reset_index()

    # Create the figure and set size
    plt.figure(figsize=(12, 6))

    # Create bar plot showing opioid deaths per year
    sns.barplot(x='Year', y='Opioid_Death', data=opioid_deaths_by_year, color='skyblue')

    # Calculate average number of opioid deaths
    average_deaths = opioid_deaths_by_year['Opioid_Death'].mean()

    # Add horizontal line to represent average deaths per year
    plt.axhline(y=average_deaths, color='red', linestyle='--', label=f'Average: {average_deaths:.2f}')

    # Set plot title and labels
    plt.title('Opioid-Related Deaths Per Year', fontsize=14)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Number of Opioid Deaths', fontsize=12)

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha="right")

    # Show the legend with the average line
    plt.legend()

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plot
    plt.show()

# --- Visualization 2: Distribution of Deaths by Age Bin ---
def visualize_deaths_by_age_bin(df):
    """
    Visualizes the distribution of deaths across age bins, highlighting the bin with maximum deaths.

    Args:
        df (pd.DataFrame): Input DataFrame with 'Age_Bin' column.
    """
    # Check if 'Age_Bin' column exists in the DataFrame
    if 'Age_Bin' not in df.columns:
         print("Error: 'Age_Bin' column missing, cannot plot this chart")
         return

    # Count deaths in each 'Age_Bin' and sort index
    age_bin_counts = df['Age_Bin'].value_counts().sort_index()

    # Check if age bin data is empty
    if age_bin_counts.empty:
      print("No data for Age_Bin to plot")
      return

    # Assign a gradient color palette based on number of age bins
    colors = sns.color_palette("viridis", len(age_bin_counts))

    # Create figure and set figure size
    plt.figure(figsize=(10, 6))

    # Create bar chart for death distribution
    bars = plt.bar(age_bin_counts.index, age_bin_counts.values, color=colors)

    # Find the age bin with maximum deaths
    max_count = age_bin_counts.max()
    max_bin = age_bin_counts.idxmax()

    # Annotate bar with the maximum death count
    plt.text(max_bin, max_count, f'Max: {max_count}', ha='center', va='bottom', color='black', fontweight = 'bold')

    # Set chart title and labels
    plt.title('Distribution of Deaths by Age Bin', fontsize=14)
    plt.xlabel('Age Bin', fontsize=12)
    plt.ylabel('Number of Deaths', fontsize=12)

    # Set x-axis ticks to age bin indexes
    plt.xticks(age_bin_counts.index)

    # Create legend labels with original bin ranges
    bin_labels = [f'Age Bin {i*10}-{((i+1)*10)-1}' for i in age_bin_counts.index]

    # Add the legend and position to avoid clutter
    plt.legend(bars, bin_labels, title='Age Bins', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Adjust layout to prevent overlapping
    plt.tight_layout()

    # Display the plot
    plt.show()

# --- Visualization 3: Proportion of Opioid vs Non-Opioid Deaths by Sex ---
def visualize_opioid_deaths_by_sex(df):
    """
    Visualizes the proportion of opioid vs non-opioid deaths by sex with annotations.

    Args:
       df (pd.DataFrame): Input DataFrame with 'Sex' and 'Opioid_Death' columns.
    """
    # Check if required columns are in df
    if 'Sex' not in df.columns or 'Opioid_Death' not in df.columns:
       print("Error: Missing 'Sex' or 'Opioid_Death' column, cannot plot this chart.")
       return

    # Group data by 'Sex' and 'Opioid_Death', and unstack for plotting proportions
    opioid_sex = df.groupby(['Sex', 'Opioid_Death']).size().unstack(fill_value=0)

    # Handle case where the grouped data is empty
    if opioid_sex.empty:
        print("No data to plot")
        return

    # Calculate proportions of opioid vs non-opioid deaths for each sex
    opioid_sex_proportions = opioid_sex.div(opioid_sex.sum(axis=1), axis=0)

    # Create figure and set size
    plt.figure(figsize=(8, 6))

    # Create stacked bar chart for opioid death proportions by sex
    bars = opioid_sex_proportions.plot(kind='bar', stacked=True, color=['pink', 'skyblue'])

    # Add percentage labels to each bar segment
    for bar in bars.patches:
       width = bar.get_width()
       height = bar.get_height()
       x, y = bar.get_xy()
       plt.text(x + width/2, y+ height/2, f'{height:.1%}', ha='center', va='center', color='black', fontweight = 'bold')

    # Set plot title and labels
    plt.title('Proportion of Opioid vs Non-Opioid Deaths by Sex', fontsize=14)
    plt.xlabel('Sex (0: Female, 1: Male)', fontsize=12)
    plt.ylabel('Proportion of Deaths', fontsize=12)

    # Remove x-axis rotation
    plt.xticks(rotation=0)

    # Add legend and position to avoid clutter
    plt.legend(title='Opioid Death (0: No, 1: Yes)', labels=['Non-Opioid', 'Opioid'],bbox_to_anchor=(1.05, 1), loc='upper left')

    # Adjust layout to prevent overlapping
    plt.tight_layout()

    # Display the plot
    plt.show()

# --- Visualization 4: Top 10 Cities by Opioid Deaths ---
def visualize_top_cities_opioid_deaths(df):
    """
    Visualizes the top 10 cities by opioid deaths with annotations.

    Args:
        df (pd.DataFrame): Input DataFrame with 'Residence City' and 'Opioid_Death' columns.
    """
    # Check if the required columns are in DataFrame
    if 'Residence City' not in df.columns or 'Opioid_Death' not in df.columns:
        print("Error: Missing 'Residence City' or 'Opioid_Death' column, cannot plot this chart")
        return

    # Group by city and count total opioid deaths, and get the top 10 cities
    city_deaths = df.groupby('Residence City')['Opioid_Death'].sum().nlargest(10).sort_values()
    if city_deaths.empty:
        print("No data to plot")
        return

    # Create a figure and set size
    plt.figure(figsize=(10, 7))

    # Create horizontal bar chart for top 10 cities
    bars = city_deaths.plot(kind='barh', color=sns.color_palette("Greens", len(city_deaths)))

    # Add the number of deaths on each bar
    for bar in bars.patches:
       width = bar.get_width()
       x, y = bar.get_xy()
       plt.text(width, y + bar.get_height() / 2, f'{int(width)}', va = 'center', fontweight = 'bold')

    # Set plot title and labels
    plt.title('Top 10 Cities by Opioid-Related Deaths', fontsize=14)
    plt.xlabel('Number of Opioid Deaths', fontsize=12)
    plt.ylabel('City', fontsize=12)

    # Adjust layout to prevent overlapping
    plt.tight_layout()

    # Display the plot
    plt.show()

# --- Visualization 5: Race/Ethnicity and Opioid Death ---
def visualize_race_ethnicity_opioid_deaths(df):
    """
    Visualizes the distribution of opioid deaths by race/ethnicity.

    Args:
        df (pd.DataFrame): Input DataFrame with 'Race_Ethnicity_Encoded' and 'Opioid_Death' columns.
    """
    # Check if columns exist in the DataFrame
    if 'Race_Ethnicity_Encoded' not in df.columns or 'Opioid_Death' not in df.columns:
      print("Error: Missing 'Race_Ethnicity_Encoded' or 'Opioid_Death' column, cannot plot this chart.")
      return

    # Group data by Race/Ethnicity and count total opioid deaths, sort the values
    race_ethnicity_deaths = df.groupby('Race_Ethnicity_Encoded')['Opioid_Death'].sum().sort_values()
    if race_ethnicity_deaths.empty:
      print("No data to plot")
      return

    # Create a rainbow color palette based on number of race/ethnicities
    colors = sns.color_palette('rainbow', len(race_ethnicity_deaths))

    # Create figure and set size
    plt.figure(figsize=(10, 6))

    # Create bar chart for opioid deaths per race/ethnicity
    bars = plt.bar(race_ethnicity_deaths.index, race_ethnicity_deaths.values, color = colors)

    # Set plot title and labels
    plt.title('Opioid Deaths by Race/Ethnicity', fontsize=14)
    plt.xlabel('Race/Ethnicity (Encoded)', fontsize=12)
    plt.ylabel('Number of Opioid Deaths', fontsize=12)

    # Set x-axis ticks
    plt.xticks(race_ethnicity_deaths.index)

    # Create legend with race/ethnicity labels based on mapping
    race_labels = {
          0: 'Other_Unknown',
          1: 'Black_Not Hispanic/Latino',
          2: 'Asian_Not Hispanic/Latino',
          3: 'White_Not Hispanic/Latino',
    }

    labels = [race_labels.get(i, f'Race/Ethnicity {i}') for i in race_ethnicity_deaths.index]

    plt.legend(bars, labels, title = 'Race/Ethnicity', bbox_to_anchor=(1.05, 1), loc='upper left')

    # Adjust layout to prevent overlapping
    plt.tight_layout()

    # Display the plot
    plt.show()

# --- Example calls to the visualization functions
if 'Year' in df_final.columns and 'Opioid_Death' in df_final.columns:
  visualize_opioid_deaths_by_year(df_final)
else:
  print("Error: 'Year' or 'Opioid_Death' column missing, cannot plot this chart")
visualize_deaths_by_age_bin(df_final)
visualize_opioid_deaths_by_sex(df_final)
visualize_top_cities_opioid_deaths(df_final)
visualize_race_ethnicity_opioid_deaths(df_final)

3.2 Detailed visual analysis of major drugs

In [ ]:
import matplotlib.patches as mpatches

# --- Visualization 1: Drug Deaths by Year (Fentanyl, Heroin, Cocaine) using filled areas ---
# Ensure necessary columns are present
if not all(col in df_final.columns for col in ['Year', 'Fentanyl', 'Heroin', 'Cocaine']):
    print("Error: Missing 'Year', 'Fentanyl', 'Heroin', or 'Cocaine' column, cannot plot this chart.")
else:
    # Group by year and count each drug
    drug_deaths_by_year = df_final.groupby('Year')[['Fentanyl', 'Heroin', 'Cocaine']].sum().reset_index()

    # Create the figure and set size
    plt.figure(figsize=(12, 6))

    # Plot filled areas with transparency for each drug
    plt.plot(drug_deaths_by_year['Year'], drug_deaths_by_year['Fentanyl'], color='red', label='Fentanyl')
    plt.fill_between(drug_deaths_by_year['Year'], drug_deaths_by_year['Fentanyl'], color='red', alpha=0.3)

    plt.plot(drug_deaths_by_year['Year'], drug_deaths_by_year['Heroin'], color='blue', label='Heroin')
    plt.fill_between(drug_deaths_by_year['Year'], drug_deaths_by_year['Heroin'], color='blue', alpha=0.2)

    plt.plot(drug_deaths_by_year['Year'], drug_deaths_by_year['Cocaine'], color='green', label='Cocaine')
    plt.fill_between(drug_deaths_by_year['Year'], drug_deaths_by_year['Cocaine'], color='green', alpha=0.2)

    # Calculate and plot average for fentanyl
    average_fentanyl = drug_deaths_by_year['Fentanyl'].mean()
    plt.axhline(y=average_fentanyl, color='red', linestyle='--', label=f'Avg Fentanyl: {average_fentanyl:.2f}')

    # Set plot title and labels
    plt.title('Drug-Related Deaths Per Year', fontsize=14)
    plt.xlabel('Year', fontsize=12)
    plt.ylabel('Number of Deaths', fontsize=12)

    # Rotate x-axis labels for better readability
    plt.xticks(rotation=45, ha="right")

    # Show the legend with the average line
    plt.legend()

    # Adjust layout to prevent overlap
    plt.tight_layout()

    # Display the plot
    plt.show()

# --- Visualization 2: Distribution of Deaths by Age Bin ---
# Check if 'Age_Bin' column exists in the DataFrame
if 'Age_Bin' not in df_final.columns:
    print("Error: 'Age_Bin' column missing, cannot plot this chart")
else:
    # Count deaths in each 'Age_Bin' and sort index
    age_bin_counts = df_final['Age_Bin'].value_counts().sort_index()

    # Check if age bin data is empty
    if age_bin_counts.empty:
      print("No data for Age_Bin to plot")
    else:
        # Assign a gradient color palette based on number of age bins
        colors = sns.color_palette("viridis", len(age_bin_counts))

        # Create figure and set figure size
        plt.figure(figsize=(10, 6))

        # Create bar chart for death distribution
        bars = plt.bar(age_bin_counts.index, age_bin_counts.values, color=colors)

        # Find the age bin with maximum deaths
        max_count = age_bin_counts.max()
        max_bin = age_bin_counts.idxmax()

        # Annotate bar with the maximum death count
        plt.text(max_bin, max_count, f'Max: {max_count}', ha='center', va='bottom', color='black', fontweight = 'bold')

        # Set chart title and labels
        plt.title('Distribution of Deaths by Age Bin', fontsize=14)
        plt.xlabel('Age Bin', fontsize=12)
        plt.ylabel('Number of Deaths', fontsize=12)

        # Set x-axis ticks to age bin indexes
        plt.xticks(age_bin_counts.index)

        # Create legend labels with original bin ranges
        bin_labels = [f'Age Bin {i*10}-{((i+1)*10)-1}' for i in age_bin_counts.index]

        # Add the legend and position to avoid clutter
        plt.legend(bars, bin_labels, title='Age Bins', bbox_to_anchor=(1.05, 1), loc='upper left')

        # Adjust layout to prevent overlapping
        plt.tight_layout()

        # Display the plot
        plt.show()

# --- Visualization 3: Drug Deaths by Sex (Fentanyl, Heroin, Cocaine) using bars with opacity ---
# Check if required columns are in df
if not all(col in df_final.columns for col in ['Sex', 'Fentanyl', 'Heroin', 'Cocaine']):
        print("Error: Missing 'Sex', 'Fentanyl', 'Heroin', or 'Cocaine' column, cannot plot this chart.")
else:
    # Group data by sex and sum each drug
    drug_sex = df_final.groupby('Sex')[['Fentanyl', 'Heroin', 'Cocaine']].sum().reset_index()

    # Handle case where the grouped data is empty
    if drug_sex.empty:
        print("No data to plot")
    else:
        # Create figure and set size
        plt.figure(figsize=(8, 6))

        # Bar plot for Fentanyl
        plt.bar(drug_sex['Sex'], drug_sex['Fentanyl'], color='red', alpha=0.7, label='Fentanyl')
        # Bar plot for Heroin, slightly offset
        plt.bar(drug_sex['Sex'] + 0.2, drug_sex['Heroin'], color='blue', alpha=0.5, label='Heroin', width=0.2)
        # Bar plot for Cocaine, slightly offset
        plt.bar(drug_sex['Sex'] + 0.4, drug_sex['Cocaine'], color='green', alpha=0.5, label='Cocaine', width=0.2)

        # Set plot title and labels
        plt.title('Drug-Related Deaths by Sex', fontsize=14)
        plt.xlabel('Sex (0: Female, 1: Male)', fontsize=12)
        plt.ylabel('Number of Deaths', fontsize=12)

        # Remove x-axis rotation and set specific ticks
        plt.xticks([0, 1])

        # Add legend and position to avoid clutter
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

        # Adjust layout to prevent overlapping
        plt.tight_layout()

        # Display the plot
        plt.show()

# --- Visualization 4: Top 10 Cities by Drug Deaths (Fentanyl, Heroin, Cocaine) with different transparency of bars ---
# Check if the required columns are in DataFrame
if not all(col in df_final.columns for col in ['Residence City', 'Fentanyl', 'Heroin', 'Cocaine']):
    print("Error: Missing 'Residence City', 'Fentanyl', 'Heroin', or 'Cocaine' column, cannot plot this chart")
else:
    # Group by city and sum each drug
    city_deaths = df_final.groupby('Residence City')[['Fentanyl', 'Heroin', 'Cocaine']].sum()

    # Get top 10 cities by sum of all drugs
    city_deaths['Total'] = city_deaths.sum(axis = 1)
    top_cities = city_deaths.nlargest(10, 'Total')
    top_cities = top_cities.sort_values(by = 'Total') #Sort for plotting

    if top_cities.empty:
        print("No data to plot")
    else:
        # Create a figure and set size
        plt.figure(figsize=(10, 7))

        # Create bar chart with transparency
        bar_width = 0.2
        index = range(len(top_cities))

        plt.barh(index, top_cities['Fentanyl'], color='red', alpha=0.7, label = 'Fentanyl', height = bar_width)
        plt.barh([i + bar_width for i in index], top_cities['Heroin'], color='blue', alpha=0.5, label = 'Heroin', height = bar_width)
        plt.barh([i + 2 * bar_width for i in index], top_cities['Cocaine'], color='green', alpha=0.5, label = 'Cocaine', height = bar_width)

        # Set plot title and labels
        plt.title('Top 10 Cities by Drug-Related Deaths', fontsize=14)
        plt.xlabel('Number of Deaths', fontsize=12)
        plt.ylabel('City', fontsize=12)

        # Set y-ticks to be the city names
        plt.yticks([i + bar_width for i in index], top_cities.index)

        # Add legend and position to avoid clutter
        plt.legend()

        # Adjust layout to prevent overlapping
        plt.tight_layout()

        # Display the plot
        plt.show()

# --- Visualization 5: Race/Ethnicity and Drug Death (Fentanyl, Heroin, Cocaine) using bars with different levels of transparency ---
# Check if columns exist in the DataFrame
if not all(col in df_final.columns for col in ['Race_Ethnicity_Encoded', 'Fentanyl', 'Heroin', 'Cocaine']):
    print("Error: Missing 'Race_Ethnicity_Encoded', 'Fentanyl', 'Heroin', or 'Cocaine' column, cannot plot this chart.")
else:
    # Group data by race/ethnicity and sum each drug
    race_ethnicity_deaths = df_final.groupby('Race_Ethnicity_Encoded')[['Fentanyl', 'Heroin', 'Cocaine']].sum().sort_values(by='Fentanyl')
    if race_ethnicity_deaths.empty:
      print("No data to plot")
    else:
        # Create a figure and set size
        plt.figure(figsize=(12, 8))

        # Bar plot for Fentanyl
        bar_width = 0.2
        index = range(len(race_ethnicity_deaths))
        plt.barh(index, race_ethnicity_deaths['Fentanyl'], color='red', alpha=0.7, label='Fentanyl', height=bar_width)
        plt.barh([i + bar_width for i in index], race_ethnicity_deaths['Heroin'], color='blue', alpha=0.5, label='Heroin', height=bar_width)
        plt.barh([i + 2 * bar_width for i in index], race_ethnicity_deaths['Cocaine'], color='green', alpha=0.5, label='Cocaine', height=bar_width)


        # Set plot title and labels
        plt.title('Drug-Related Deaths by Race/Ethnicity', fontsize=14)
        plt.xlabel('Number of Deaths', fontsize=12)
        plt.ylabel('Race/Ethnicity', fontsize=12)

        # Create labels for race/ethnicity
        race_ethnicity_labels = {
            0: 'Asian_Hispanic/Latino',
            1: 'Asian_Not Hispanic/Latino',
            2: 'Asian_Unknown',
            3: 'Black_Hispanic/Latino',
            4: 'Black_Not Hispanic/Latino',
            5: 'Black_Unknown',
            6: 'Other_Hispanic/Latino',
            7: 'Other_Not Hispanic/Latino',
            8: 'Other_Unknown',
             9: 'White_Hispanic/Latino',
            10: 'White_Not Hispanic/Latino',
            11: 'White_Unknown',
        }

        # Set y-ticks and labels
        plt.yticks([i + bar_width for i in index], [race_ethnicity_labels.get(i, f'Unknown {i}') for i in race_ethnicity_deaths.index])

        # Create legend without color patches
        plt.legend()

        # Adjust layout to prevent overlapping
        plt.tight_layout()

        # Display the plot
        plt.show()